# CSCI203 Assignment 1 — Stage 1: One-Teller Bank Simulation

This notebook implements the **single-teller** version first. It follows the assignment rules:

- discrete-event simulation (the clock jumps to the next arrival/completion event),
- one common priority queue,
- priority `3 > 2 > 1`,
- for equal priority, earlier arrival comes first,
- the dummy record `0 0` stops file reading,
- no `heapq`, `queue.PriorityQueue`, `collections`, or other data-structure libraries,
- only the imports shown by the assignment are used in the program cells,
- all required service/queue/idle statistics are calculated.

> For one teller, Round Robin is trivial because Teller 0 is the only possible teller. We will extend the same design to 2, 3, and 4 tellers later.


## 2. Imports

I only used `sys` and `numpy` following the example instructions.

In [ ]:
import sys
import numpy


## 3. Customer-record positions

Each customer is stored in a normal Python list:

`[arrival_time, service_time, priority, original_order]`

The constants below make the code easier to read.


In [ ]:
ARRIVAL = 0
SERVICE = 1
PRIORITY = 2
ORDER = 3


## 4. Read the input file

The file is read record by record. The last `0 0` record is a dummy record and is **not** stored as a customer.


In [ ]:
def read_customers(filename):
    customers = []
    sequence_number = 0
    previous_arrival = -1.0

    try:
        input_file = open(filename, "r")
    except OSError:
        print("ERROR: Could not open file:", filename)
        sys.exit(1)

    for raw_line in input_file:
        line = raw_line.strip()

        # Ignore completely blank lines.
        if line == "":
            continue

        parts = line.split()

        # Every useful line must contain at least arrival and service time.
        if len(parts) < 2:
            input_file.close()
            raise ValueError("Invalid input line: " + line)

        arrival_time = float(parts[0])
        service_time = float(parts[1])

        # The dummy record may contain only the two zeros.
        if arrival_time == 0.0 and service_time == 0.0:
            break

        if len(parts) < 3:
            input_file.close()
            raise ValueError("Customer record is missing its priority: " + line)

        priority = int(parts[2])

        if priority < 1 or priority > 3:
            input_file.close()
            raise ValueError("Priority must be 1, 2, or 3: " + line)

        # The assignment says records are already sorted by arrival time.
        if arrival_time < previous_arrival:
            input_file.close()
            raise ValueError("Input records are not in ascending arrival-time order.")

        customer = [arrival_time, service_time, priority, sequence_number]
        customers.append(customer)

        previous_arrival = arrival_time
        sequence_number += 1

    input_file.close()
    return customers


## 5. Implement the priority queue ourselves

The queue is a normal list, but i have manually inserted each new waiting customer in the correct position.

Ordering rule:

1. higher priority first;
2. if priorities are equal, earlier arrival first;
3. if arrival and priority are both equal, preserve original file order.


In [ ]:
def customer_should_come_before(customer_a, customer_b):
    # Higher priority comes first.
    if customer_a[PRIORITY] > customer_b[PRIORITY]:
        return True
    if customer_a[PRIORITY] < customer_b[PRIORITY]:
        return False

    # Same priority: earlier arrival comes first.
    if customer_a[ARRIVAL] < customer_b[ARRIVAL]:
        return True
    if customer_a[ARRIVAL] > customer_b[ARRIVAL]:
        return False

    # Exact tie: preserve original input order.
    return customer_a[ORDER] < customer_b[ORDER]


def priority_enqueue(queue, customer):
    # Find where the customer belongs.
    insert_position = len(queue)

    for i in range(len(queue)):
        if customer_should_come_before(customer, queue[i]):
            insert_position = i
            break

    # Make one extra space at the end.
    queue.append(customer)

    # Shift existing elements one position to the right.
    i = len(queue) - 1
    while i > insert_position:
        queue[i] = queue[i - 1]
        i -= 1

    queue[insert_position] = customer


def priority_dequeue(queue):
    # The first element is always the next customer to serve.
    first_customer = queue[0]

    # Shift remaining customers one place to the left.
    for i in range(1, len(queue)):
        queue[i - 1] = queue[i]

    queue.pop()
    return first_customer


## 6. One-teller discrete-event simulation

the simulation clock starts at **time 0.0**. Therefore any time before the first customer arrives counts as teller idle time.

If an arrival and a completion have exactly the same timestamp, this notebook processes the **completion first**. With the provided continuous-valued sample data, an exact tie is very unlikely.


In [ ]:
def simulate_one_teller(customers):
    waiting_queue = []

    current_time = 0.0
    next_customer_index = 0
    number_of_customers = len(customers)

    teller_busy = False
    teller_finish_time = float("inf")

    customers_served = 0
    teller_busy_time = 0.0
    total_service_time = 0.0
    total_waiting_time = 0.0
    maximum_queue_length = 0

    # Continue until:
    # 1) no future arrivals remain,
    # 2) the queue is empty, and
    # 3) the teller is no longer busy.
    while (next_customer_index < number_of_customers or
           len(waiting_queue) > 0 or
           teller_busy):

        if next_customer_index < number_of_customers:
            next_arrival_time = customers[next_customer_index][ARRIVAL]
        else:
            next_arrival_time = float("inf")

        if teller_busy:
            next_completion_time = teller_finish_time
        else:
            next_completion_time = float("inf")

        # -------------------------------------------------------------
        # EVENT TYPE 1: teller completes the current service.
        # -------------------------------------------------------------
        if next_completion_time <= next_arrival_time:
            current_time = next_completion_time
            teller_busy = False
            teller_finish_time = float("inf")

            # If somebody is already waiting, start the first queued
            # customer's service immediately at this same time.
            if len(waiting_queue) > 0:
                customer = priority_dequeue(waiting_queue)

                waiting_time = current_time - customer[ARRIVAL]
                total_waiting_time += waiting_time

                total_service_time += customer[SERVICE]
                teller_busy_time += customer[SERVICE]
                customers_served += 1

                teller_busy = True
                teller_finish_time = current_time + customer[SERVICE]

        # -------------------------------------------------------------
        # EVENT TYPE 2: a new customer arrives.
        # -------------------------------------------------------------
        else:
            current_time = next_arrival_time
            customer = customers[next_customer_index]
            next_customer_index += 1

            if not teller_busy:
                # Immediate service: waiting time is zero.
                total_service_time += customer[SERVICE]
                teller_busy_time += customer[SERVICE]
                customers_served += 1

                teller_busy = True
                teller_finish_time = current_time + customer[SERVICE]
            else:
                # Teller is busy, so the customer joins the one
                # common priority queue.
                priority_enqueue(waiting_queue, customer)

                if len(waiting_queue) > maximum_queue_length:
                    maximum_queue_length = len(waiting_queue)

    # The last event processed is the final service completion.
    total_simulation_time = current_time

    if customers_served > 0:
        average_service_time = total_service_time / customers_served
        average_waiting_time = total_waiting_time / customers_served
    else:
        average_service_time = 0.0
        average_waiting_time = 0.0

    if total_simulation_time > 0.0:
        average_queue_length = total_waiting_time / total_simulation_time
        teller_idle_time = total_simulation_time - teller_busy_time

        # Protect against tiny negative floating-point roundoff.
        if teller_idle_time < 0.0 and teller_idle_time > -0.000000001:
            teller_idle_time = 0.0

        teller_idle_rate = teller_idle_time / total_simulation_time
    else:
        average_queue_length = 0.0
        teller_idle_time = 0.0
        teller_idle_rate = 0.0

    results = {
        "customers_served": customers_served,
        "total_simulation_time": total_simulation_time,
        "average_service_time": average_service_time,
        "average_waiting_time": average_waiting_time,
        "maximum_queue_length": maximum_queue_length,
        "average_queue_length": average_queue_length,
        "teller_busy_time": teller_busy_time,
        "teller_idle_time": teller_idle_time,
        "teller_idle_rate": teller_idle_rate,
        "total_waiting_time": total_waiting_time,
        "total_service_time": total_service_time
    }

    return results


## 7. Print the required statistics

The extra busy/idle/total-queueing values are printed as useful checks. The assignment-required values are clearly labelled.


In [ ]:
def print_statistics(results):
    print("\n================ ONE-TELLER RESULTS ================")
    print("Number of customers served by Teller 0 :", results["customers_served"])
    print("Total simulation time                  : {:.4f}".format(results["total_simulation_time"]))
    print("Average service time per customer      : {:.4f}".format(results["average_service_time"]))
    print("Average waiting time per customer      : {:.4f}".format(results["average_waiting_time"]))
    print("Maximum length of queue                :", results["maximum_queue_length"])
    print("Average length of queue                : {:.4f}".format(results["average_queue_length"]))
    print("Idle rate of Teller 0                  : {:.4f} ({:.2f}%)".format(
        results["teller_idle_rate"],
        results["teller_idle_rate"] * 100.0
    ))

    print("\n---------------- CHECKING VALUES --------------------")
    print("Total customer queuing/waiting time    : {:.4f}".format(results["total_waiting_time"]))
    print("Total teller busy time                 : {:.4f}".format(results["teller_busy_time"]))
    print("Total teller idle time                 : {:.4f}".format(results["teller_idle_time"]))
    print("======================================================")


## 8. Input

For this one-teller notebook, the number of tellers and input file are fixed.

The simulation always runs with:
- Number of tellers = 1
- Input file = a1-sample.txt


In [ ]:
# For one-teller simulation

number_of_tellers = 1

# User provides only the input file name
filename = input("Enter input filename for one teller: ").strip()

customers = read_customers(filename)

print("Loaded", len(customers), "customers from", filename)
print("Number of tellers:", number_of_tellers)

Loaded 100 customers from a1-sample.txt
Number of tellers: 1


## 9. Run the simulation and record notebook execution runtime
- **Total simulation time** = simulated bank time from time 0 until the last customer leaves.
- **Wall time** = how long your computer actually took to execute this notebook cell.

As the assignment explicitly requires the simulation statistics. The wall-time measurement is included because you asked to record program runtime too.

In [ ]:
%%time
results = simulate_one_teller(customers)
print_statistics(results)



================ ONE-TELLER RESULTS ================
Number of customers served by Teller 0 : 100
Total simulation time                  : 1282.5675
Average service time per customer      : 12.7983
Average waiting time per customer      : 361.9065
Maximum length of queue                : 57
Average length of queue                : 28.2173
Idle rate of Teller 0                  : 0.0021 (0.21%)

---------------- CHECKING VALUES --------------------
Total customer queuing/waiting time    : 36190.6483
Total teller busy time                 : 1279.8287
Total teller idle time                 : 2.7388
CPU times: total: 0 ns
Wall time: 833 μs
